# 1. CNN(Convolutional Neural Network)이란

이미지, 음성, 영상 등 **공간적(위치적) 패턴**이 있는 데이터를 처리하는 데 특화된 인공 신경망입니다.  
MLP(다층 퍼셉트론)와 달리 입력 데이터의 **공간 구조를 유지하며 특징을 추출**합니다.  
이미지 분류, 객체 탐지, 얼굴 인식 등 컴퓨터 비전 분야에서 **표준적으로 사용**됩니다.

---

# 2. CNN의 기본 구조

```
입력 이미지
 ├─ [ 컨볼루션층(Conv) + 활성화 함수(ReLU) ]
 ├─ [ 풀링층(Pooling) ]
 ├─ (Conv + Pooling 반복)
 ├─ [ Flatten ]
 ├─ [ 완전 연결층(Dense) ]
 └─ 출력층 (분류 결과)
```

---

# 3. 컨볼루션층 (Convolution Layer)

**역할**  
입력 이미지에서 **국소적인 특징(에지, 코너, 텍스처 등)** 을 **필터(커널)** 를 사용해 추출

**작동 방식**  
작은 크기의 필터가 이미지 위를 슬라이딩하며 동일 위치 픽셀과 곱셈 후 더하여 **특징맵(feature map)** 을 만듬

**장점**  
위치 정보와 형태 정보를 유지하면서 파라미터 수를 줄일 수 있고, 여러 필터로 다양한 패턴을 탐지

---

# 4. 풀링층 (Pooling Layer)

**역할**  
컨볼루션층에서 추출한 특징맵의 **공간적 크기를 줄여 연산량을 감소**시키고 중요한 특징을 강조하여 모델의 **일반화 성능을 높임**.

**종류**  
- **Max Pooling**: 윈도우 내에서 최대값을 선택  
- **Average Pooling**: 윈도우 내에서 평균값을 선택

**효과**  
풀링은 이미지의 작은 이동이나 회전에 강인해지도록 도와주며 **과적합(overfitting)** 을 줄이는 효과

---

# 5. CNN의 일반적인 흐름

1. **Conv Layer**: 필터로 로컬 특징 추출  
2. **ReLU 활성화 함수**: 비선형성을 추가  
3. **Pooling Layer**: 특징맵의 크기를 축소하여 중요한 정보만 유지  
4. **반복**: Conv + Pooling을 여러 번 반복하여 저수준 → 고수준 특징을 단계적으로 학습  
5. **Flatten**: 2D 특징맵을 1D 벡터로 변환  
6. **Fully Connected(Dense)**: 추출된 특징을 바탕으로 최종 분류 수행

---

# 6. 구성 요소 요약

| 구성 요소              | 역할                                       |
| ---------------------- | ------------------------------------------ |
| **컨볼루션층 (Conv)**   | 국소적인 특징 추출 (에지, 형태 등)           |
| **풀링층 (Pooling)**   | 특징맵 크기 축소, 중요한 정보 유지, 불변성 확보 |
| **활성화 함수 (ReLU)** | 비선형성 부여                               |
| **완전 연결층 (Dense)** | 추출된 특징을 바탕으로 최종 예측             |

---

# 7. CNN 구조 예시

```
입력 이미지 (32x32x3)
 ├─ Conv2D(32 filters, 3x3) + ReLU
 ├─ MaxPooling2D(2x2)
 ├─ Conv2D(64 filters, 3x3) + ReLU
 ├─ MaxPooling2D(2x2)
 ├─ Flatten
 ├─ Dense(128) + ReLU
 ├─ Dense(10) + Softmax
```

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow.keras.datasets as ds

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# [1] MNIST 데이터셋 불러오기 (손글씨 숫자 이미지)
(x_train, y_train), (x_test, y_test) = ds.mnist.load_data()

# [2] 입력 데이터 전처리
# (60000, 28, 28) → (60000, 28, 28, 1) 로 reshape (CNN은 4D 입력 필요)
x_train = x_train.reshape(60000, 28, 28, 1)
x_test = x_test.reshape(10000, 28, 28, 1)

# 픽셀값 [0, 255] → [0, 1] 로 정규화
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# 레이블을 One-hot 인코딩 (0~9 → 벡터)
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# [3] CNN 모델 정의 (LeNet-5 유사 구조)
cnn = Sequential()

# 입력층 → 첫 컨볼루션층 (6개의 5x5 필터, same padding)
cnn.add(Conv2D(6, (5, 5), padding='same', activation='relu', input_shape=(28, 28, 1)))

# 첫 풀링층 (2x2 Max Pooling, stride=2)
cnn.add(MaxPooling2D(pool_size=(2, 2), strides=2))

# 두 번째 컨볼루션층 (16개의 5x5 필터, valid padding)
cnn.add(Conv2D(16, (5, 5), padding='valid', activation='relu'))

# 두 번째 풀링층 (2x2 Max Pooling, stride=2)
cnn.add(MaxPooling2D(pool_size=(2, 2), strides=2))

# 세 번째 컨볼루션층 (120개의 5x5 필터, valid padding)
cnn.add(Conv2D(120, (5, 5), padding='valid', activation='relu'))

# Fully Connected로 연결하기 위해 1D 벡터로 펼침
cnn.add(Flatten())

# 완전 연결층 (Dense), 84개 유닛, ReLU 활성화
cnn.add(Dense(units=84, activation='relu'))

# 출력층: 10개 클래스, softmax 활성화로 확률 출력
cnn.add(Dense(units=10, activation='softmax'))

# [4] 모델 컴파일
# 손실 함수: 다중 클래스 → categorical_crossentropy
# 옵티마이저: Adam
# 평가지표: 정확도
cnn.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# [5] 모델 학습
# 배치크기: 128, epoch: 30, 검증 데이터: x_test, y_test
cnn.fit(
    x_train, y_train,
    batch_size=128,
    epochs=30,
    validation_data=(x_test, y_test),
    verbose=2
)

# [6] 모델 평가 (테스트 데이터 정확도)
res = cnn.evaluate(x_test, y_test, verbose=0)
print('정확률 =', res[1] * 100)


Epoch 1/30
469/469 - 3s - loss: 0.2910 - accuracy: 0.9139 - val_loss: 0.0799 - val_accuracy: 0.9756 - 3s/epoch - 7ms/step
Epoch 2/30
469/469 - 1s - loss: 0.0776 - accuracy: 0.9763 - val_loss: 0.0566 - val_accuracy: 0.9829 - 751ms/epoch - 2ms/step
Epoch 3/30
469/469 - 1s - loss: 0.0539 - accuracy: 0.9836 - val_loss: 0.0419 - val_accuracy: 0.9865 - 760ms/epoch - 2ms/step
Epoch 4/30
469/469 - 1s - loss: 0.0439 - accuracy: 0.9863 - val_loss: 0.0521 - val_accuracy: 0.9834 - 781ms/epoch - 2ms/step
Epoch 5/30
469/469 - 1s - loss: 0.0345 - accuracy: 0.9892 - val_loss: 0.0342 - val_accuracy: 0.9888 - 760ms/epoch - 2ms/step
Epoch 6/30
469/469 - 1s - loss: 0.0306 - accuracy: 0.9907 - val_loss: 0.0334 - val_accuracy: 0.9896 - 755ms/epoch - 2ms/step
Epoch 7/30
469/469 - 1s - loss: 0.0271 - accuracy: 0.9909 - val_loss: 0.0308 - val_accuracy: 0.9911 - 749ms/epoch - 2ms/step
Epoch 8/30
469/469 - 1s - loss: 0.0211 - accuracy: 0.9934 - val_loss: 0.0375 - val_accuracy: 0.9888 - 752ms/epoch - 2ms/step
Epo

| 부분                        | 설명                         |
| ------------------------- | --------------------------------- |
| `input_shape=(28, 28, 1)` | MNIST 데이터가 28×28 흑백 이미지라 채널=1     |
| `(5, 5)` 필터               | 작은 국소 영역(5×5)에서 특징 추출, LeNet-5 전통 |
| 출력 채널=6                   | 서로 다른 패턴을 6개 필터로 동시에 감지           |
| `padding='same'`          | 출력 이미지 크기를 입력과 동일하게 유지            |
| `activation='relu'`       | 비선형성, 학습 효율, gradient 소실 방지       |